In [1]:
!pip install scikit-surprise gensim nltk --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 20.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np

In [3]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-25m.zip
!unzip -q ml-25m.zip
print("MovieLens downloaded")

MovieLens downloaded


In [7]:
movies = pd.read_csv('ml-25m/movies.csv')
links = pd.read_csv('ml-25m/links.csv')
ratings = pd.read_csv('ml-25m/ratings.csv')
tags = pd.read_csv('ml-25m/tags.csv')
tmdb_movies = pd.read_csv('tmdb_5000_movies.csv')
tmdb_credits = pd.read_csv('tmdb_5000_credits.csv')

In [8]:
print("MovieLens movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Tags:", tags.shape)
print("Links:", links.shape)
print("TMDB movies:", tmdb_movies.shape)
print("TMDB credits:", tmdb_credits.shape)

MovieLens movies: (62423, 3)
Ratings: (25000095, 4)
Tags: (1093360, 4)
Links: (62423, 3)
TMDB movies: (4803, 20)
TMDB credits: (4803, 4)


In [9]:
movies.head(3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance


In [10]:
ratings.head(3)

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828


In [11]:
tags.head(3)

,userId,movieId,tag,timestamp
0,3,260,classic,1439472355
1,3,260,sci-fi,1439472256
2,4,1732,dark comedy,1573943598


In [12]:
links.head(3)

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0


In [13]:
tmdb_movies.head(3)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466


In [14]:
tmdb_credits.head(3)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


In [15]:
tmdb_movies.columns.tolist()

['budget',
 'genres',
 'homepage',
 'id',
 'keywords',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'production_companies',
 'production_countries',
 'release_date',
 'revenue',
 'runtime',
 'spoken_languages',
 'status',
 'tagline',
 'title',
 'vote_average',
 'vote_count']

In [16]:
tmdb_credits.columns.tolist()

['movie_id', 'title', 'cast', 'crew']

In [17]:
tmdb_credits['cast'][0]

'[{"cast_id": 242, "character": "Jake Sully", "credit_id": "5602a8a7c3a3685532001c9a", "gender": 2, "id": 65731, "name": "Sam Worthington", "order": 0}, {"cast_id": 3, "character": "Neytiri", "credit_id": "52fe48009251416c750ac9cb", "gender": 1, "id": 8691, "name": "Zoe Saldana", "order": 1}, {"cast_id": 25, "character": "Dr. Grace Augustine", "credit_id": "52fe48009251416c750aca39", "gender": 1, "id": 10205, "name": "Sigourney Weaver", "order": 2}, {"cast_id": 4, "character": "Col. Quaritch", "credit_id": "52fe48009251416c750ac9cf", "gender": 2, "id": 32747, "name": "Stephen Lang", "order": 3}, {"cast_id": 5, "character": "Trudy Chacon", "credit_id": "52fe48009251416c750ac9d3", "gender": 1, "id": 17647, "name": "Michelle Rodriguez", "order": 4}, {"cast_id": 8, "character": "Selfridge", "credit_id": "52fe48009251416c750ac9e1", "gender": 2, "id": 1771, "name": "Giovanni Ribisi", "order": 5}, {"cast_id": 7, "character": "Norm Spellman", "credit_id": "52fe48009251416c750ac9dd", "gender": 

In [18]:
tmdb_credits['crew'][0]

'[{"credit_id": "52fe48009251416c750aca23", "department": "Editing", "gender": 0, "id": 1721, "job": "Editor", "name": "Stephen E. Rivkin"}, {"credit_id": "539c47ecc3a36810e3001f87", "department": "Art", "gender": 2, "id": 496, "job": "Production Design", "name": "Rick Carter"}, {"credit_id": "54491c89c3a3680fb4001cf7", "department": "Sound", "gender": 0, "id": 900, "job": "Sound Designer", "name": "Christopher Boyes"}, {"credit_id": "54491cb70e0a267480001bd0", "department": "Sound", "gender": 0, "id": 900, "job": "Supervising Sound Editor", "name": "Christopher Boyes"}, {"credit_id": "539c4a4cc3a36810c9002101", "department": "Production", "gender": 1, "id": 1262, "job": "Casting", "name": "Mali Finn"}, {"credit_id": "5544ee3b925141499f0008fc", "department": "Sound", "gender": 2, "id": 1729, "job": "Original Music Composer", "name": "James Horner"}, {"credit_id": "52fe48009251416c750ac9c3", "department": "Directing", "gender": 2, "id": 2710, "job": "Director", "name": "James Cameron"},

In [19]:
import ast

def extract_names(text):
    return [i['name'] for i in ast.literal_eval(text)][:5]

def extract_director(text):
    return [i['name'] for i in ast.literal_eval(text) if i['job'] == 'Director']

In [20]:
tmdb = tmdb_movies.merge(tmdb_credits,left_on='id',right_on='movie_id')

In [21]:
tmdb.shape

(4803, 24)

In [22]:
tmdb_movies.shape

(4803, 20)

In [23]:
tmdb_credits.shape

(4803, 4)

In [24]:
tmdb.columns.to_list()

['budget',
 'genres',
 'homepage',
 'id',
 'keywords',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'production_companies',
 'production_countries',
 'release_date',
 'revenue',
 'runtime',
 'spoken_languages',
 'status',
 'tagline',
 'title_x',
 'vote_average',
 'vote_count',
 'movie_id',
 'title_y',
 'cast',
 'crew']

In [25]:
tmdb.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,spoken_languages,status,tagline,title_x,vote_average,vote_count,movie_id,title_y,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


I don't need so many columns for recommendation. Some of them are irrelevant so I will drop them.


In [26]:
tmdb = tmdb[['id', 'title_x', 'overview', 'genres', 'keywords', 'cast', 'crew']] # I had the title column in both datasets but it wasn't the merge key so now i have got title_x and title_y

In [27]:
tmdb = tmdb.rename(columns={'title_x': 'title'})

In [28]:
tmdb.head(2)

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


With this I have removed the irrelevant columns.

In [29]:
tmdb['genres'] = tmdb['genres'].apply(extract_names)
tmdb['keywords'] = tmdb['keywords'].apply(extract_names)
tmdb['cast'] = tmdb['cast'].apply(extract_names)
tmdb['crew'] = tmdb['crew'].apply(extract_director)

In [30]:
tmdb.head(2)

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weave...",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley, ...",[Gore Verbinski]


In [31]:
movie_details = tmdb[['title', 'genres', 'cast', 'crew']].copy()
movie_details.head(2)

,title,genres,cast,crew
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[Sam Worthington, Zoe Saldana, Sigourney Weave...",[James Cameron]
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[Johnny Depp, Orlando Bloom, Keira Knightley, ...",[Gore Verbinski]


In [32]:
tmdb['genres'] = tmdb['genres'].apply(lambda x: [i.replace(' ','') for i in x])
tmdb['keywords'] = tmdb['keywords'].apply(lambda x: [i.replace(' ','') for i in x])
tmdb['cast'] = tmdb['cast'].apply(lambda x: [i.replace(' ','') for i in x])
tmdb['crew'] = tmdb['crew'].apply(lambda x: [i.replace(' ','') for i in x])

In [33]:
tmdb.head(2)

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver, ...",[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley, Ste...",[GoreVerbinski]


In [34]:
tmdb['overview'] = tmdb['overview'].fillna('')

In [35]:
tmdb['tags'] = tmdb.apply(lambda x: (x['overview'] + ' ' + ' '.join(x['genres']) + ' ' + ' '.join(x['keywords']) + ' ' + ' '.join(x['cast']) + ' ' + ' '.join(x['crew'])).lower(), axis=1)

In [36]:
tmdb['tags'][0]

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society samworthington zoesaldana sigourneyweaver stephenlang michellerodriguez jamescameron'

In [37]:
tmdb = tmdb[['id', 'title', 'tags']]
tmdb.head(2)

,id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."


In [38]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [39]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
def preprocess(text):
   words = [i for i in text.split() if i not in stop_words]
   lst = [lemmatizer.lemmatize(w) for w in words]
   return ' '.join(lst)

In [40]:
tmdb['tags'] = tmdb['tags'].apply(preprocess)

In [41]:
tmdb['tags']

,tags
0,"22nd century, paraplegic marine dispatched moo..."
1,"captain barbossa, long believed dead, come bac..."
2,cryptic message bond’s past sends trail uncove...
3,"following death district attorney harvey dent,..."
4,"john carter war-weary, former military captain..."
...,...
4798,el mariachi want play guitar carry family trad...
4799,newlywed couple's honeymoon upended arrival re...
4800,"""signed, sealed, delivered"" introduces dedicat..."
4801,ambitious new york attorney sam sent shanghai ...


In [42]:
links.head(2)

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0


In [43]:
tmdb_ml = links.merge(tmdb, left_on='tmdbId', right_on='id')

In [44]:
print(tmdb_ml.shape)
tmdb_ml.head(2)

(4602, 6)


,movieId,imdbId,tmdbId,id,title,tags
0,1,114709,862.0,862,Toy Story,"led woody, andy's toy live happily room andy's..."
1,10,113189,710.0,710,GoldenEye,james bond must unmask mysterious head janus s...


In [45]:
tmdb_ml = tmdb_ml[['movieId', 'title', 'tags']]
tmdb_ml.head(2)

,movieId,title,tags
0,1,Toy Story,"led woody, andy's toy live happily room andy's..."
1,10,GoldenEye,james bond must unmask mysterious head janus s...


In [46]:
ratings.head(2)

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817


In [47]:
rated_movies = ratings[ratings['movieId'].isin(tmdb_ml['movieId'])]
print("Ratings for our movies:", rated_movies.shape)
print("Unique movies with ratings:", rated_movies['movieId'].nunique())

Ratings for our movies: (17258140, 4)
Unique movies with ratings: 4595


In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf.fit_transform(tmdb_ml['tags'])
print(tfidf_matrix.shape)

(4602, 5000)


In [49]:
from sklearn.metrics.pairwise import cosine_similarity
tfidf_similarity = cosine_similarity(tfidf_matrix)
print(tfidf_similarity.shape)

(4602, 4602)


In [50]:
# Get index of Avatar
idx = tmdb_ml[tmdb_ml['title'] == 'Avatar'].index[0]

# Get similarity scores for Avatar
scores = list(enumerate(tfidf_similarity[idx]))

# Sort by similarity
scores = sorted(scores, key=lambda x: x[1], reverse=True)

# Print top 5 most similar movies
for i, score in scores[1:6]:
    print(tmdb_ml.iloc[i]['title'], "—", round(score, 3))

Apollo 18 — 0.188
Aliens — 0.172
The Book of Life — 0.166
The Adventures of Pluto Nash — 0.15
The Time Machine — 0.132


In [51]:
def get_tfidf_recommendations(movie, n=5):
    matches = tmdb_ml[tmdb_ml['title'] == movie]
    if matches.empty:
        return f"Movie '{movie}' not found in dataset"

    idx = matches.index[0]
    scores = list(enumerate(tfidf_similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    recommendations = []
    for i, score in scores[1:n+1]:
        recommendations.append((tmdb_ml.iloc[i]['title'], round(score, 3)))

    return recommendations

In [52]:
get_tfidf_recommendations('Avatar')


[('Apollo 18', np.float64(0.188)),
 ('Aliens', np.float64(0.172)),
 ('The Book of Life', np.float64(0.166)),
 ('The Adventures of Pluto Nash', np.float64(0.15)),
 ('The Time Machine', np.float64(0.132))]

In [53]:
import gensim.downloader as api
glove = api.load('glove-wiki-gigaword-100')
print("GloVe loaded")

[==================================================] 100.0% 128.1/128.1MB downloaded
GloVe loaded


In [54]:
def get_glove_vector(tags):
  words = tags.split();
  vectors = [glove[word] for word in words if word in glove]
  if(not vectors):
    return np.zeros(100)
  return np.mean(vectors,axis=0)


In [55]:
get_glove_vector(tmdb_ml['tags'][0])

array([ 9.53063741e-02,  2.44149461e-01,  2.37337768e-01, -2.99454689e-01,
        7.66464919e-02,  3.06096554e-01, -1.33558944e-01, -4.65533175e-02,
        3.71578299e-02, -1.54453486e-01, -6.52549341e-02,  8.46517012e-02,
        1.44892409e-01,  2.74255350e-02,  1.93056487e-03, -2.85877883e-02,
        1.55143157e-01,  1.95330098e-01, -1.13849767e-01,  3.60904813e-01,
        1.72019765e-01, -1.18873775e-01,  6.98239356e-02, -1.27983630e-01,
        3.28727633e-01,  1.47431388e-01, -6.43054619e-02, -9.26156938e-02,
        1.17284760e-01,  4.93030921e-02, -2.84651339e-01,  4.29261774e-01,
        1.70621574e-01, -4.28822823e-02, -2.84707528e-02,  1.30518764e-01,
        6.71872124e-02,  2.45412975e-03,  6.19587973e-02, -3.21513206e-01,
       -5.83609268e-02, -8.96673426e-02,  2.69355662e-02, -1.68489218e-01,
       -5.85873798e-02, -1.17682153e-04, -2.54739523e-01,  2.59780344e-02,
        2.39020497e-01, -4.30664390e-01, -8.48105252e-02, -2.58556396e-01,
        7.67283663e-02,  

In [56]:
glove_matrix = tmdb_ml['tags'].apply(get_glove_vector)


In [57]:
glove_matrix.shape


(4602,)

In [58]:
glove_matrix = np.vstack(glove_matrix)
print(glove_matrix.shape)

(4602, 100)


In [59]:
glove_similarity = cosine_similarity(glove_matrix)
print(glove_similarity.shape)

(4602, 4602)


In [60]:
def get_glove_recommendations(movie, n=5):
    matches = tmdb_ml[tmdb_ml['title'] == movie]
    if matches.empty:
        return f"Movie '{movie}' not found in dataset"

    idx = matches.index[0]
    scores = list(enumerate(glove_similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    recommendations = []
    for i, score in scores[1:n+1]:
        recommendations.append((tmdb_ml.iloc[i]['title'], round(score, 3)))

    return recommendations


In [61]:
get_glove_recommendations('Batman Begins')

[('The Dark Knight', np.float32(0.95)),
 ('A Most Violent Year', np.float32(0.944)),
 ('The International', np.float32(0.941)),
 ('Batman Returns', np.float32(0.941)),
 ('Zipper', np.float32(0.941))]

In [62]:
get_tfidf_recommendations('Batman Begins')

[('The Dark Knight', np.float64(0.354)),
 ('The Dark Knight Rises', np.float64(0.353)),
 ('Batman', np.float64(0.235)),
 ('Batman Returns', np.float64(0.235)),
 ('Batman Forever', np.float64(0.178))]

In [63]:
def get_content_recommendations(movie, n =5, alpha = 0.5):
  matches = tmdb_ml[tmdb_ml['title'] == movie]
  if(matches.empty):
    return f"Movie '{movie}' not found"
  idx = matches.index[0]
  tfidf_scores = tfidf_similarity[idx]
  glove_scores = glove_similarity[idx]

  tfidf_norm = (tfidf_scores - tfidf_scores.min()) / (tfidf_scores.max() - tfidf_scores.min())
  glove_norm = (glove_scores - glove_scores.min()) / (glove_scores.max() - glove_scores.min())

  combined = alpha * tfidf_norm + (1 - alpha) * glove_norm

  scores = list(enumerate(combined))
  scores = sorted(scores, key=lambda x: x[1], reverse=True)

  return [(tmdb_ml.iloc[i]['title'], round(float(score), 3)) for i, score in scores[1:n+1]]

In [64]:
print("Avatar:")
print(get_content_recommendations('Avatar'))

print("\nBatman Begins:")
print(get_content_recommendations('Batman Begins'))

Avatar:
[('Aliens', 0.493), ('Titan A.E.', 0.472), ('The Inhabited Island', 0.465), ('The Book of Life', 0.464), ('Starship Troopers', 0.46)]

Batman Begins:
[('The Dark Knight', 0.631), ('The Dark Knight Rises', 0.616), ('Batman Returns', 0.563), ('Batman', 0.532), ('Defendor', 0.507)]


In [65]:
import pickle
pickle.dump(tfidf_similarity, open('tfidf_similarity.pkl', 'wb'))
pickle.dump(glove_similarity, open('glove_similarity.pkl', 'wb'))
pickle.dump(tmdb_ml, open('movies.pkl', 'wb'))
print("Content model saved")

Content model saved


In [66]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

import os
os.makedirs('/content/drive/MyDrive/movie_recommender/models', exist_ok=True)

shutil.copy('tfidf_similarity.pkl', '/content/drive/MyDrive/movie_recommender/models/')
shutil.copy('glove_similarity.pkl', '/content/drive/MyDrive/movie_recommender/models/')
shutil.copy('movies.pkl', '/content/drive/MyDrive/movie_recommender/models/')

print("Models saved to Drive")

Mounted at /content/drive
Models saved to Drive


In [67]:
print(rated_movies['rating'].min())
print(rated_movies['rating'].max())

0.5
5.0


In [68]:
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(rated_movies[['userId', 'movieId', 'rating']], reader)

In [69]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
print("Training ratings:", trainset.n_ratings)
print("Test ratings:", len(testset))

Training ratings: 13806512
Test ratings: 3451628


In [70]:
svd = SVD(n_factors=100, random_state=42)
svd.fit(trainset)
print("SVD trained")

SVD trained


In [71]:
from surprise import accuracy
predictions = svd.test(testset)
rmse = accuracy.rmse(predictions)
print(f"RMSE: {rmse}")

RMSE: 0.7819
RMSE: 0.7819198110221363


In [72]:
# Predict what user 1 would rate Avatar (movieId = 1449)
pred = svd.predict(uid=1, iid=1449)
print(f"Predicted rating: {pred.est}")

Predicted rating: 4.272386184167407


In [73]:
def get_svd_recommendations(user_id, n=5):
    rated = rated_movies[rated_movies['userId'] == user_id]['movieId'].values
    unrated = tmdb_ml[~tmdb_ml['movieId'].isin(rated)]['movieId'].values
    predictions = [(movie_id, svd.predict(user_id, movie_id).est) for movie_id in unrated]
    predictions = sorted(predictions, key=lambda x: x[1], reverse=True)

    results = []
    for movie_id, score in predictions[:n]:
        title = tmdb_ml[tmdb_ml['movieId'] == movie_id]['title'].values[0]
        results.append((title, round(score, 3)))

    return results

In [74]:
get_svd_recommendations(user_id=1)

[('The Godfather', np.float64(4.945)),
 ('The Godfather: Part II', np.float64(4.846)),
 ('The Secret in Their Eyes', np.float64(4.796)),
 ('Central Station', np.float64(4.784)),
 ('The Lives of Others', np.float64(4.765))]

In [75]:
pickle.dump(svd, open('svd_model.pkl', 'wb'))
print("SVD model saved")

SVD model saved


In [76]:
shutil.copy('svd_model.pkl', '/content/drive/MyDrive/movie_recommender/models/')
print("SVD model saved to Drive")

SVD model saved to Drive


In [77]:
def get_hybrid_recommendations(user_id, movie, n=5, alpha=0.5):

    content_recs = get_content_recommendations(movie, n=20)


    hybrid_scores = []
    for title, content_score in content_recs:

        movie_row = tmdb_ml[tmdb_ml['title'] == title]
        if movie_row.empty:
            continue
        movie_id = movie_row['movieId'].values[0]


        svd_score = svd.predict(user_id, movie_id).est


        final_score = alpha * content_score + (1 - alpha) * (svd_score / 5.0)
        hybrid_scores.append((title, round(final_score, 3)))


    hybrid_scores = sorted(hybrid_scores, key=lambda x: x[1], reverse=True)
    return hybrid_scores[:n]

In [78]:
get_hybrid_recommendations(user_id=1, movie='Avatar')

[('Cloud Atlas', np.float64(0.624)),
 ('E.T. the Extra-Terrestrial', np.float64(0.596)),
 ('The Book of Life', np.float64(0.593)),
 ('Aliens', np.float64(0.592)),
 ('Star Trek Into Darkness', np.float64(0.589))]

In [79]:
print("Content only:")
print(get_content_recommendations('Avatar'))

print("\nSVD only:")
print(get_svd_recommendations(user_id=1))

print("\nHybrid:")
print(get_hybrid_recommendations(user_id=1, movie='Avatar'))

Content only:
[('Aliens', 0.493), ('Titan A.E.', 0.472), ('The Inhabited Island', 0.465), ('The Book of Life', 0.464), ('Starship Troopers', 0.46)]

SVD only:
[('The Godfather', np.float64(4.945)), ('The Godfather: Part II', np.float64(4.846)), ('The Secret in Their Eyes', np.float64(4.796)), ('Central Station', np.float64(4.784)), ('The Lives of Others', np.float64(4.765))]

Hybrid:
[('Cloud Atlas', np.float64(0.624)), ('E.T. the Extra-Terrestrial', np.float64(0.596)), ('The Book of Life', np.float64(0.593)), ('Aliens', np.float64(0.592)), ('Star Trek Into Darkness', np.float64(0.589))]


In [80]:
tmdb.head(2)

,id,title,tags
0,19995,Avatar,"22nd century, paraplegic marine dispatched moo..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed dead, come bac..."


In [82]:
def explain_recommendation(movie1, movie2):
    m1 = movie_details[movie_details['title'] == movie1].iloc[0]
    m2 = movie_details[movie_details['title'] == movie2].iloc[0]

    reasons = []

    # Common genres
    common_genres = set(m1['genres']) & set(m2['genres'])
    if common_genres:
        reasons.append(f"Similar genres: {', '.join(common_genres)}")

    # Common cast
    common_cast = set(m1['cast']) & set(m2['cast'])
    if common_cast:
        reasons.append(f"Shared actors: {', '.join(common_cast)}")

    # Common director
    common_crew = set(m1['crew']) & set(m2['crew'])
    if common_crew:
        reasons.append(f"Same director: {', '.join(common_crew)}")

    if not reasons:
        reasons.append("Similar themes and style")

    return reasons

In [83]:
explain_recommendation('Avatar', 'Aliens')

['Similar genres: Action, Science Fiction',
 'Shared actors: Sigourney Weaver',
 'Same director: James Cameron']

In [84]:
def get_recommendations_with_explanation(user_id, movie, n=5):
    recs = get_hybrid_recommendations(user_id, movie, n=n)

    results = []
    for title, score in recs:
        reasons = explain_recommendation(movie, title)
        results.append({
            'title': title,
            'score': score,
            'reasons': reasons
        })

    return results

In [85]:
recs = get_recommendations_with_explanation(user_id=1, movie='Avatar')
for r in recs:
    print(f"\n🎬 {r['title']} (score: {r['score']})")
    for reason in r['reasons']:
        print(f"   → {reason}")


🎬 Cloud Atlas (score: 0.624)
   → Similar genres: Science Fiction

🎬 E.T. the Extra-Terrestrial (score: 0.596)
   → Similar genres: Adventure, Fantasy, Science Fiction

🎬 The Book of Life (score: 0.593)
   → Similar genres: Adventure
   → Shared actors: Zoe Saldana

🎬 Aliens (score: 0.592)
   → Similar genres: Action, Science Fiction
   → Shared actors: Sigourney Weaver
   → Same director: James Cameron

🎬 Star Trek Into Darkness (score: 0.589)
   → Similar genres: Adventure, Action, Science Fiction
   → Shared actors: Zoe Saldana


In [86]:
pickle.dump(tmdb_ml, open('movies.pkl', 'wb'))
pickle.dump(movie_details, open('movie_details.pkl', 'wb'))
pickle.dump(tfidf_similarity, open('tfidf_similarity.pkl', 'wb'))
pickle.dump(glove_similarity, open('glove_similarity.pkl', 'wb'))
pickle.dump(svd, open('svd_model.pkl', 'wb'))
print("✅ All models saved")

✅ All models saved


In [87]:
import shutil
files = ['movies.pkl', 'movie_details.pkl', 'tfidf_similarity.pkl', 'glove_similarity.pkl', 'svd_model.pkl']
for f in files:
    shutil.copy(f, f'/content/drive/MyDrive/movie_recommender/models/{f}')
print("All models saved to Drive")

All models saved to Drive


In [88]:
!pip install huggingface_hub
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [89]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = "vaibhav343/movie-recommender-models"

api.upload_file(path_or_fileobj="tfidf_similarity.pkl",
                path_in_repo="tfidf_similarity.pkl",
                repo_id=repo_id)

api.upload_file(path_or_fileobj="glove_similarity.pkl",
                path_in_repo="glove_similarity.pkl",
                repo_id=repo_id)

api.upload_file(path_or_fileobj="svd_model.pkl",
                path_in_repo="svd_model.pkl",
                repo_id=repo_id)

print("Large models uploaded to HF Hub")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  tfidf_similarity.pkl        :   0%|          |  567kB /  169MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  glove_similarity.pkl        :  11%|#1        | 9.71MB / 84.7MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  svd_model.pkl               :   2%|2         | 11.9MB /  529MB            

Large models uploaded to HF Hub
